<H1>Importing datasets</H1>

In [1]:
from datasets import load_dataset

dataset = load_dataset("8Opt/multilingual-classification-0001")



d:\jup\codespaces-jupyter\.venv313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset["train"][0]

{'text': 'यो फिल्म को न कथा ठिक छ न ब्याक म्युजिक ठिक छ पहाड र तराई त्यसै पनी मिलेर बसेको छ यो अनाआवश्य को फिल्म बनाएर के नाटक गर्या होला',
 'lang': 'nep',
 'label': 11}

In [3]:
print(dataset["train"].column_names)

['text', 'lang', 'label']


In [4]:
df = dataset["train"].to_pandas()
df.shape

(25942, 3)

<h1>Text preprocessing</h1>

In [5]:
from gensim.models import Word2Vec
import emoji
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
import re
import string

In [6]:
def textLower(text):
    return text.lower()

def remove_url(text):
    pattern = re.compile(r'https??://\S+|www\.\S+')
    return pattern.sub(r'',text)

exclude = string.punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('','',exclude))

def remove_emoticons(text):
    pattern = r'[:;=8xX][\-^]?[)(/\\DPpOo*]'
    return re.sub(pattern,'',text)

def remove_emoji(text):
    return emoji.replace_emoji(text, replace='')

def clean_text(text):
    return re.sub(r'\s+',' ',text).strip()

In [7]:
df["processed_text"] = df["text"].apply(remove_emoji)
df["processed_text"] = df["processed_text"].apply(textLower)
df["processed_text"] = df["processed_text"].apply(remove_url)
df["processed_text"] = df["processed_text"].apply(remove_punctuation)
df["processed_text"] = df["processed_text"].apply(clean_text)

In [8]:
df["tokens"] = df["processed_text"].apply(word_tokenize)
df["tokens"]

0        [यो, फिल्म, को, न, कथा, ठिक, छ, न, ब्याक, म्यु...
1        [التوتر, الطويل, الامد, بيزيد, الاكتئاب, والقل...
2        [zumal, ja, noch, der, verdacht, im, raum, ste...
3        [من, امروز, رفتم, خرید, چیزی, که, به, چشم, دید...
4                    [shegiya, wannan, barauniya, ce, url]
                               ...                        
25937    [challengetomuslims, almajiri, children, are, ...
25938    [بیشتر, از, ده, ساله, یه, دل, سیر, با, پدر, ما...
25939    [ملحد, ارتقا, کو, اتنا, پسند, کرتا, ہے, کہ, مج...
25940    [kıbrıs, barış, harekâtının, yıldönümünde, ece...
25941    [user, yan, tasha, da, ghetto, aka, tara, kawa...
Name: tokens, Length: 25942, dtype: object

<h1>Splitting the data</h1>


In [9]:
from sklearn.model_selection import train_test_split

In [10]:
train_df, temp_df = train_test_split(
    df,
    test_size= 0.3,
    random_state=42,
    stratify=df["lang"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size= 0.5,
    random_state=42,
    stratify=temp_df["lang"]
)

In [11]:
train_df.shape

(18159, 5)

In [12]:
test_df.shape

(3892, 5)

<h1>Word to vec</h1>

In [13]:
w2vmodel = Word2Vec(
    sentences=train_df["tokens"].to_list(),
    vector_size=100,
    min_count=2,
    workers=4
)

In [14]:
w2vmodel.wv.most_similar("फिल्म", topn=10)

[('कुरा', 0.9987553358078003),
 ('नेपाल', 0.9987533092498779),
 ('दिन', 0.9985290169715881),
 ('भन्ने', 0.998394787311554),
 ('र', 0.9981058835983276),
 ('लाई', 0.9980022311210632),
 ('पनि', 0.9979023933410645),
 ('कारण', 0.9978317618370056),
 ('थे', 0.9977517127990723),
 ('छ', 0.9977104067802429)]

In [15]:
w2vmodel.wv.similarity("word","फिल्म")

np.float32(0.61746925)

<H1>Sentence Vector</H1>

In [16]:
import numpy as np

In [17]:
def sentence_vectors(tokens, w2vmodel, max_len=50):
    vectors = []

    for word in tokens:
        if word in w2vmodel.wv:
            vectors.append(w2vmodel.wv[word])

    # Limit sentence to 50 words
    vectors = vectors[:max_len]

    # Padding
    while len(vectors) < max_len:
        vectors.append(np.zeros(w2vmodel.vector_size))

    return np.array(vectors)

In [18]:
tokens = train_df["tokens"].iloc[0]

vector = sentence_vectors(tokens, w2vmodel)

print(vector)
print(vector.shape)

[[-0.0095489   0.01953405  0.01552935 ... -0.02151484  0.00314552
  -0.01543963]
 [-0.04516488  0.51330966  0.39552072 ... -0.74078512 -0.17021316
  -0.68199843]
 [-0.00394413  0.01466348  0.00391856 ... -0.01903421  0.0031804
  -0.01975989]
 ...
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]
(50, 100)


In [19]:
X_train = np.array([
    sentence_vectors(tokens, w2vmodel)
    for tokens in train_df["tokens"]
])
X_val = np.array([
    sentence_vectors(tokens, w2vmodel)
    for tokens in val_df["tokens"]
])
X_test = np.array([
    sentence_vectors(tokens, w2vmodel)
    for tokens in test_df["tokens"]
])

In [20]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(18159, 50, 100)
(3891, 50, 100)
(3892, 50, 100)


In [21]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_train = encoder.fit_transform(train_df["lang"])
y_val = encoder.transform(val_df["lang"])
y_test = encoder.transform(test_df["lang"])

In [22]:
y_test

array([ 2,  0,  7, ...,  7, 11,  7], shape=(3892,))

In [23]:
print(encoder.classes_)

['amh' 'arb' 'deu' 'eng' 'fas' 'hau' 'hin' 'nep' 'spa' 'tur' 'urd' 'zho']


<h1>Training</h1>

In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [25]:
model = Sequential([
    LSTM(128, input_shape=(50,100)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(12, activation="softmax")
])


d:\jup\codespaces-jupyter\.venv313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [26]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [27]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
568/568 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.3861 - loss: 1.5664 - val_accuracy: 0.5050 - val_loss: 1.2674
Epoch 2/10
568/568 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.5552 - loss: 1.0872 - val_accuracy: 0.5551 - val_loss: 1.0895
Epoch 3/10
568/568 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.6454 - loss: 0.8710 - val_accuracy: 0.7232 - val_loss: 0.6467
Epoch 4/10
568/568 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.6933 - loss: 0.8642 - val_accuracy: 0.8098 - val_loss: 0.6091
Epoch 5/10
568/568 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.8248 - loss: 0.5259 - val_accuracy: 0.8777 - val_loss: 0.3361
Epoch 6/10
568/568 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.8709 - loss: 0.3988 - val_accuracy: 0.9029 - val_loss: 0.2971
Epoch 7/10
568/568 ━━━━━━━━━━━━━━━━━━━━ 28s 49ms/step - accuracy: 0.9026 - loss: 0.2991 - val_accuracy: 0.9396 - val_loss: 0.2038
Epoch 8/10
568/568 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - accuracy: 0.9420 - loss: 0.2269 - val_a

In [28]:
model.evaluate(X_test,y_test)

122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9712 - loss: 0.1082


[0.10816410183906555, 0.971222996711731]

In [31]:
text = input("Enter your text:")
text_tokens = word_tokenize(text)
text_vectors = sentence_vectors(text_tokens, w2vmodel, max_len=50)
text_vectors = np.expand_dims(text_vectors, axis=0)
pred = model.predict(text_vectors)
predicted_class = np.argmax(pred, axis=1)[0]
predicted_lang = encoder.inverse_transform([predicted_class])[0]
confidence = pred[0][predicted_class]
print(f"Predicted language: {predicted_lang} \nConfidence: {confidence:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Predicted language: spa 
Confidence: 1.00
